# Lossless two-fin inverse design — reflection exceptional points

Find **second-order EPs in the reflection Jones matrices** of a reciprocal, lossless, periodic two-fin dielectric meta-atom at $\lambda=633$ nm. This notebook runs **alongside** `TwoFin_Lossless_Transmission_EP.ipynb` and the separate absorbing/full-$4\times4$ EP experiment; neither is modified.

The physical material index is **strictly real**, $n_\mathrm{fin}=1.76+0i$; the substrate index is $n_\mathrm{sub}=1.46$. The RCWA solver retains the complete, complex, power-normalized $4\times4$ scattering matrix

$$S=\begin{pmatrix}R_{11}&T_{12}\\T_{21}&R_{22}\end{pmatrix},\qquad S^\dagger S=I,\quad S=S^T.$$

The *individual* $2\times2$ reflection blocks are generally contractive and can be **nonnormal** even though $S$ is unitary. Thus reflection EPs do not require material absorption. Unlike the two transmission directions, reciprocity **does not** equate $R_{11}$ and $R_{22}$: simultaneous reflection EPs are **two independent complex degeneracy constraints** and may be substantially harder than finding one EP. Set `TARGET='both'` to search jointly, or `'R11'` / `'R22'` to investigate each side.

We use the fixed laboratory circular basis $U=\frac1{\sqrt2}\begin{pmatrix}1&1\\i&-i\end{pmatrix}$, $R_\circ=U^\dagger R_{xy}U$. The basis change is a unitary similarity, so eigenvalues, defects, eigenvector overlap, and EP existence do not depend on choosing linear versus circular polarization. **Lab circular labels are not directional helicity labels.**

> Optimization identifies *candidates*, not proofs of physical EPs. Hard-edge RCWA convergence, Jordan-defect tests, full-S conservation, and optional eigenvalue-sheet exchange are required before making claims.

## 1 · Set up lossless simulation and GPU workload

Run in the same repository directory as `engine_rcwa.py`, `engine_geometry.py`, `engine_inverse.py`, `engine_transmission_ep.py`, `twofin.py` and the new `engine_reflection_ep.py`. Only the RCWA solver is used by this notebook; the existing FDTD engine remains for validation.

In [ ]:
# Shared engines live in ../core (see ../PROJECTS.md). Run with the notebook folder as cwd.
import sys; from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / 'core').resolve()))

In [ ]:
from dataclasses import asdict
from pathlib import Path
import hashlib, json, time
import numpy as np
import matplotlib.pyplot as plt
import torch

import engine_rcwa, engine_geometry, engine_inverse, engine_transmission_ep, engine_reflection_ep, twofin
from twofin import Config, draw_geometry, write_gds, stokes
from engine_rcwa import Settings as RCWASettings, solve as rcwa_solve, solve_torch
from engine_inverse import BOUNDS, FIXED_PORTS, enforce_unit_cell
from engine_reflection_ep import (
    VERSION, KEYS, TARGETS, circular_unitary, reflection_blocks,
    discrim_numpy, block_diagnostics, loss_torch, to_tensors,
    simulate_lossless, diagnostics, report, screen_candidate,
    generate_seeds, adam_refine, physical_config, save_result,
    load_result, track_pair, loop_exchange,
)

OUT = Path('results/lossless_reflection_ep')
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGET = 'both'       # 'both', 'R11' (bottom incidence), or 'R22' (top incidence)
assert TARGET in TARGETS

RUN_GRADIENT_TEST = True
RUN_OPTIMIZATION = True
FORCE_OPTIMIZATION = False
RUN_HARD_VERIFICATION = True
RUN_HARD_POLISH = False
RUN_EP_MAP = False           # Optional: 49 additional binary solves
RUN_ENCIRCLING = False      # Optional: 41 additional binary solves

N_SCREEN, KEEP = 16, 3
COARSE_ITER, REFINE_ITER = 60, 40
SMOOTH_EDGE_UM = 0.006
SEED = 41

cfg = Config(wavelength=0.633, period=0.400, n_sub=1.46,
             n_fin=1.76, n_fin_kappa=0.0)
cfg.validate()
assert np.isrealobj(cfg.n_fin) and cfg.n_fin_kappa == 0.0

smoke_settings = RCWASettings(order=2, raster=96, device=DEVICE)
screen_settings = RCWASettings(order=3, raster=128, device=DEVICE)
coarse_settings = RCWASettings(order=4, raster=160, device=DEVICE)
refine_settings = RCWASettings(order=5, raster=160, device=DEVICE)

print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'unavailable')
print('Target:', TARGET, '| material n:', cfg.n_fin, '+ 0i')
print('Fixed monitor planes (µm):', FIXED_PORTS)
print('Optimized geometry coordinates:', KEYS)

## 2 · EP objective and mathematical feasibility

For either $2\times2$ Jones matrix $R=\begin{pmatrix}a&b\\c&d\end{pmatrix}$, let

$$\mu=\tfrac12\mathrm{tr}R,\quad Q=R-\mu I,\quad \Delta(R)=(a-d)^2+4bc.$$

A second-order EP requires $\Delta(R)=0$ **and** $Q\neq0$, giving a defective rank-one nilpotent $Q$; the ordinary scalar degeneracy $R=\mu I$ is explicitly excluded. The optimizer minimizes a normalized $|\Delta|^2/\|R\|_F^4$, penalizes a too-small $\|Q\|_F/\|R\|_F$ and vanishing reflection, and screens unitarity/reciprocity. In joint mode it uses a smooth maximum of the **two separate** reflection objectives so it cannot ignore the harder side.

The search varies $L_i,W_i,\theta_i,x_i,y_i$ for both fins plus their shared $h$. The extinction coefficient is **never trainable** and remains exactly zero. Smooth boundaries are differentiable surrogates and must be tested against hard-edged Fourier-truncated RCWA.

In [ ]:
# Verify an EP is defective rather than a trivial repeated eigenvalue.
U = circular_unitary()
assert np.allclose(U.conj().T @ U, np.eye(2))
R_ep = .20 * np.array([[1j, 1], [1, -1j]], complex)
R_scalar = .20 * np.eye(2, dtype=complex)
assert abs(discrim_numpy(R_ep)) < 1e-12
assert abs(discrim_numpy(R_scalar)) < 1e-12
assert block_diagnostics(R_ep)['jordan_strength'] > 0.5
assert block_diagnostics(R_scalar)['jordan_strength'] < 1e-12

# A reciprocal, lossless 4x4 S can have nonorthogonal transmission eigenvectors.
rng = np.random.default_rng(17)
A = rng.normal(size=(4,4)) + 1j * rng.normal(size=(4,4))
Q, R = np.linalg.qr(A)
Q = Q @ np.diag(np.diag(R) / np.abs(np.diag(R)))
S_unitary = Q.T @ Q
assert np.allclose(S_unitary.conj().T @ S_unitary, np.eye(4))
assert np.allclose(S_unitary, S_unitary.T)
print('Lossless reciprocal example transmission overlap:',
      round(block_diagnostics(S_unitary[2:,:2])['overlap'], 3))
print('Reflection R11 and R22 have independent EP conditions; no transpose relation.')

## 3 · TORCWA differentiability smoke test

Check autograd against a central finite difference **of the same smooth objective**, before running the optimizer. Numerical gradients at nearly degenerate RCWA internal eigenmodes can be delicate.

In [ ]:
seed_guess = {k:(lo+hi)/2 for k,(lo,hi) in BOUNDS.items() if k!='kappa'}
seed_guess.update(theta1=.21, theta2=1.08, x1=-.031, y1=-.046,
                  x2=.052, y2=.039, kappa=0.0)

if RUN_GRADIENT_TEST:
    p = to_tensors(seed_guess, device=DEVICE)
    p['theta1'] = p['theta1'].clone().detach().requires_grad_(True)
    value, S_smoke, _ = simulate_lossless(cfg, smoke_settings, p,
                                           target=TARGET, edge_um=.010)
    grad = torch.autograd.grad(value, p['theta1'])[0]
    def loss_at(theta):
        trial = to_tensors({**seed_guess, 'theta1':float(theta)}, device=DEVICE)
        with torch.no_grad():
            objective, _, _ = simulate_lossless(cfg, smoke_settings, trial,
                                                target=TARGET, edge_um=.010)
        return float(objective.cpu())
    h = 2e-3
    numeric = (loss_at(seed_guess['theta1']+h) - loss_at(seed_guess['theta1']-h))/(2*h)
    analytic = float(grad.detach().cpu())
    rel_error = abs(analytic-numeric)/max(abs(analytic),abs(numeric),1e-5)
    print(f'autograd={analytic:.6g}, finite difference={numeric:.6g}, relative error={rel_error:.3g}')
    if not np.isfinite([analytic,numeric]).all() or rel_error > .30:
        raise RuntimeError('Gradient test failed; do NOT launch optimization.')
    print('Gradient smoke test passed.')

## 4 · Sobol multistart + bounded Adam

Each trial computes the full complex $S$ once. `TARGET='both'` evaluates *both reflection objectives from that same solve*. A code/configuration hash prevents inadvertently reusing an incompatible checkpoint. First use order 3 for global screening, order 4 for local optimization, and order 5 to refine the best seed.

In [ ]:
signature = {m.__name__: hashlib.sha256(Path(m.__file__).read_bytes()).hexdigest()[:12]
             for m in (engine_reflection_ep, engine_transmission_ep, engine_rcwa, engine_geometry, engine_inverse, twofin)}
run_spec = dict(version=VERSION, target=TARGET, files=signature,
    config=asdict(cfg), fixed_ports=FIXED_PORTS,
    screen=asdict(screen_settings), coarse=asdict(coarse_settings),
    refine=asdict(refine_settings), n_screen=N_SCREEN, keep=KEEP,
    iterations=(COARSE_ITER,REFINE_ITER), edge=SMOOTH_EDGE_UM, seed=SEED)
run_id = hashlib.sha256(json.dumps(run_spec,sort_keys=True).encode()).hexdigest()[:14]
checkpoint = OUT / f'{TARGET}_candidate_{run_id}.npz'
best = None
if checkpoint.exists() and not FORCE_OPTIMIZATION:
    best = load_result(checkpoint)
    print('Loaded checkpoint:', checkpoint)
elif RUN_OPTIMIZATION:
    starts, ranking = generate_seeds(cfg, screen_settings, target=TARGET,
                                     count=N_SCREEN, keep=KEEP, seed=SEED, edge_um=.010)
    print('Top screening scores:', [round(item[0],5) for item in ranking[:KEEP]])
    attempts = []
    for j, p0 in enumerate(starts, 1):
        print(f'Adam start {j}/{len(starts)} at order 4')
        try:
            attempts.append(adam_refine(cfg, coarse_settings, p0,
                target=TARGET, iterations=COARSE_ITER, lr=.045,
                edge_um=SMOOTH_EDGE_UM))
        except (RuntimeError, ValueError) as exc:
            print('Start failed:', str(exc)[:160])
    if not attempts:
        raise RuntimeError('No viable Adam start; inspect TORCWA physics/gradients.')
    attempts.sort(key=lambda item:item['loss'])
    print('Refining best start at order 5...')
    refined = adam_refine(cfg, refine_settings, attempts[0]['params'],
                          target=TARGET, iterations=REFINE_ITER, lr=.025,
                          edge_um=SMOOTH_EDGE_UM)
    best = refined if refined['loss'] < attempts[0]['loss'] else attempts[0]
    save_result(checkpoint,best,metadata=run_spec)
    print('Checkpoint:', checkpoint)
else:
    print('No optimized candidate. Enable RUN_OPTIMIZATION or restore a checkpoint.')

## 5 · Optimization history and smooth-geometry diagnostics

An EP search needs more than a minimal discriminant: report normalized eigenvalue splitting, Jones-eigenvector overlap, normalized traceless part, rank deficiency, average reflection, and conservation errors. Eigenvector overlap alone can be unstable near scalar degeneracy.

In [ ]:
if best is not None:
    print('Lowest smooth objective:', best['loss'])
    print('Geometry (µm / radians):')
    for k in KEYS:
        print(f' {k:7s}: {best["params"][k]:+.6f}')
    d_soft, _ = report(best['S'], 'Differentiable soft-edge RCWA', target=TARGET)
    history = best['history']
    fig, axes = plt.subplots(1,2,figsize=(12,4),layout='constrained')
    axes[0].semilogy(np.maximum([x['loss'] for x in history],1e-14))
    axes[0].set(xlabel='Adam iteration',ylabel='Objective',title='Optimization loss')
    for label in ('R11','R22'):
        axes[1].semilogy(np.maximum([x[f'{label}_disc'] for x in history],1e-14), label=label)
    axes[1].set(xlabel='Adam iteration',ylabel='Normalized |Δ|',title='Reflection EP proximity')
    axes[1].legend();plt.show()

## 6 · Circular-basis reflection matrices and eigenpolarizations

Plot the full complex Jones matrices, including their absolute phase at the fixed reference planes, and the eigenpolarization locations on the Poincaré sphere. For a true EP both eigenpolarizations **coalesce**, not become antipodal.

In [ ]:
if best is not None:
    R11, R22 = reflection_blocks(best['S'])
    fig, ax = plt.subplots(2,2,figsize=(10,8),layout='constrained')
    for row, (name,J) in enumerate((('R11',R11),('R22',R22))):
        im0 = ax[row,0].imshow(np.abs(J),vmin=0,cmap='magma')
        im1 = ax[row,1].imshow(np.where(np.abs(J)>1e-5,np.rad2deg(np.angle(J)),np.nan),
                               vmin=-180,vmax=180,cmap='twilight')
        for col in range(2):
            ax[row,col].set(xticks=[0,1],yticks=[0,1],
                            xticklabels=['R','L'],yticklabels=['R','L'],
                            xlabel='Incident circular channel',ylabel='Outgoing circular channel')
        ax[row,0].set_title(f'{name}: |R|');ax[row,1].set_title(f'{name}: phase (degrees)')
        fig.colorbar(im0,ax=ax[row,0]);fig.colorbar(im1,ax=ax[row,1])
    plt.show()

    from mpl_toolkits.mplot3d import Axes3D  # Register 3D projection
    U = circular_unitary()
    fig = plt.figure(figsize=(11,5))
    az = np.linspace(0,2*np.pi,36); polar=np.linspace(0,np.pi,18)
    phi,th=np.meshgrid(az,polar)
    X=np.sin(th)*np.cos(phi);Y=np.sin(th)*np.sin(phi);Z=np.cos(th)
    for k,(name,J) in enumerate((('R11',R11),('R22',R22)),1):
        a=fig.add_subplot(1,2,k,projection='3d')
        a.plot_wireframe(X,Y,Z,color='0.8',alpha=.25,lw=.5)
        eigenvalues,V=np.linalg.eig(J)
        for j in (0,1):
            lab_state=U @ (V[:,j]/np.linalg.norm(V[:,j]))
            s=stokes(lab_state)[1:]
            a.scatter(*s,s=95,label=f'λ{j+1}')
        a.set(xlim=(-1,1),ylim=(-1,1),zlim=(-1,1),xlabel='S1',ylabel='S2',
              zlabel='S3',title=f'{name} eigenpolarizations')
        a.set_box_aspect((1,1,1));a.legend()
    plt.tight_layout();plt.show()

## 7 · **Required** binary-geometry and Fourier-order validation

Round the optimized two-fin shapes to the actual polygon/GDS model and recompute at several RCWA Fourier orders. This tests *physical* geometry rather than the smooth optimization surrogate. Near an EP, eigenvectors can be extremely sensitive to tiny complex-matrix errors: inspect the complete $S$ difference between orders, not just the loss value.

In [ ]:
hard_S = {}
hard_cfg = None
if best is not None and RUN_HARD_VERIFICATION:
    hard_cfg = physical_config(cfg, best['params'])
    assert hard_cfg.n_fin_kappa == 0.0
    write_gds(hard_cfg, OUT/f'{TARGET}_binary_candidate.gds')
    hard_cache = OUT/f'{TARGET}_hard_{run_id}.npz'
    if hard_cache.exists() and not FORCE_OPTIMIZATION:
        with np.load(hard_cache,allow_pickle=False) as f:
            hard_S = {int(k.rsplit('_',1)[-1]):f[k] for k in f.files}
        print('Cached binary simulations:',hard_cache)
    else:
        for order, raster in ((5,160),(7,192),(9,224)):
            settings = RCWASettings(order=order,raster=raster,device=DEVICE,
                                     precision='complex128')
            start = time.perf_counter()
            hard_S[order] = rcwa_solve(hard_cfg,settings,ports=FIXED_PORTS)['S']
            print(f'Order {order}: {time.perf_counter()-start:.2f} s')
        np.savez_compressed(hard_cache,**{f'S_{order}':S for order,S in hard_S.items()})
    for order, S in sorted(hard_S.items()):
        report(S,f'Binary RCWA order {order}',target=TARGET)
    for low,high in ((5,7),(7,9)):
        print(f'Max full complex ΔS for order {low}→{high}: '
              f'{np.max(abs(hard_S[high]-hard_S[low])):.4g}')
    with torch.no_grad():
        soft_at5 = solve_torch(cfg,to_tensors(best['params'],device=DEVICE),
                              refine_settings,edge_um=SMOOTH_EDGE_UM,
                              ports=FIXED_PORTS).detach().cpu().numpy()
    print('Smooth-to-binary complex ΔS (order 5):',np.max(abs(soft_at5-hard_S[5])))

## 8 · Optional hard-edge Powell polishing

If smoothing moves the EP location, refine a few geometry variables using *binary* RCWA. This is off by default because every objective call is a full solve. A polished point is not automatically converged; enable high-order verification separately. The best smooth candidate is retained unchanged.

In [ ]:
polished = None
if RUN_HARD_POLISH and best is not None:
    from scipy.optimize import minimize
    base=best['params'].copy()
    keys=('theta1','height','theta2')
    bounds=[BOUNDS[k] for k in keys]
    setting=RCWASettings(order=6,raster=192,device=DEVICE,precision='complex128')
    seen=dict(loss=np.inf,params=None,S=None,evaluations=0)
    def binary_objective(x):
        trial=base.copy();trial.update({k:float(v) for k,v in zip(keys,x)})
        trial=enforce_unit_cell(trial);trial['kappa']=0.0
        try:
            S=rcwa_solve(physical_config(cfg,trial),setting,ports=FIXED_PORTS)['S']
            l,_=loss_torch(torch.as_tensor(S),target=TARGET)
            v=float(l.detach().cpu())
            seen['evaluations'] += 1
            if v < seen['loss']:
                seen.update(loss=v,params=trial.copy(),S=S.copy())
            return v
        except (RuntimeError,ValueError):
            return 100.0
    result=minimize(binary_objective,[base[k] for k in keys],method='Powell',bounds=bounds,
                    options={'maxfev':40,'xtol':1e-3,'ftol':1e-4})
    polished=seen
    print('Binary polishing:',result.message,'RCWA calls:',seen['evaluations'])
    if seen['S'] is not None:
        report(seen['S'],'Polished order-6 binary candidate',target=TARGET)
        write_gds(physical_config(cfg,seen['params']),OUT/f'{TARGET}_polished.gds')
        print('Polished design is NOT high-order verified; rerun §7 on its geometry.')

## 9 · Optional local two-parameter EP maps (both sides)

An EP is generically a codimension-two condition. We scan $\theta_1$ and $h$ around the binary candidate and inspect $|\Delta(R_{11})|/\|R_{11}\|_F^2$ and $|\Delta(R_{22})|/\|R_{22}\|_F^2$ independently. A single candidate is not guaranteed to place both zeros at the same coordinates.

In [ ]:
if RUN_EP_MAP and best is not None:
    from engine_reflection_ep import phase_scan
    center=polished['params'] if polished and polished['params'] is not None else best['params']
    settings=RCWASettings(order=7,raster=192,device=DEVICE,precision='complex128')
    xy, scans = phase_scan(cfg,settings,center,n=7)
    fig,axes=plt.subplots(1,2,figsize=(11,4.5),layout='constrained')
    for ax,name in zip(axes,('R11','R22')):
        im=ax.imshow(np.log10(np.maximum(scans[name],1e-14)),origin='lower',
                      extent=[-3,3,-.018,.018],aspect='auto')
        ax.set(xlabel='Δθ₁ (degrees)',ylabel='Δheight (µm)',
               title=f'log₁₀ normalized |Δ({name})|')
        fig.colorbar(im,ax=ax)
    plt.show()

## 10 · Optional eigenvalue-sheet exchange around a candidate

Vary two *real* geometry parameters on a closed loop. If a single second-order EP is enclosed and the eigenvalues stay distinct on the loop, continuous branch tracking should exchange the two eigenvalues after one turn. Test loop radii and Fourier order; an exchange at a poorly converged point is not a physical certificate.

In [ ]:
if RUN_ENCIRCLING and best is not None:
    base=polished['params'] if polished and polished['params'] is not None else best['params']
    settings=RCWASettings(order=7,raster=192,device=DEVICE,precision='complex128')
    phases=np.linspace(0,2*np.pi,41)
    loops={'R11':[],'R22':[]}
    for phi in phases:
        trial=base.copy()
        trial['theta1']=float(base['theta1']+np.deg2rad(2)*np.cos(phi))
        trial['height']=float(base['height']+.012*np.sin(phi))
        if not BOUNDS['theta1'][0] <= trial['theta1'] <= BOUNDS['theta1'][1]            or not BOUNDS['height'][0] <= trial['height'] <= BOUNDS['height'][1]:
            raise ValueError('Loop leaves physical bounds; reduce radius or shift center')
        trial=enforce_unit_cell(trial);trial['kappa']=0.0
        S=rcwa_solve(physical_config(cfg,trial),settings,ports=FIXED_PORTS)['S']
        A,B=reflection_blocks(S)
        loops['R11'].append(A);loops['R22'].append(B)
    fig,axes=plt.subplots(1,2,figsize=(11,4.5),layout='constrained')
    for ax,name in zip(axes,('R11','R22')):
        branches=track_pair(loops[name])
        for j in (0,1):
            ax.plot(branches[:,j].real,branches[:,j].imag,'o-',ms=2,label=f'branch {j+1}')
        swapped=loop_exchange(branches)
        ax.set(xlabel='Re(λ)',ylabel='Im(λ)',title=f'{name}: sheet exchange={swapped}')
        ax.axis('equal');ax.legend()
        print(name,'sheet exchange:',swapped)
    plt.show()

## 11 · Acceptance, phase preservation and export

Passing a numerical near-EP threshold is **not** proof of an exact EP. Require (i) nearly coalesced Jones eigenvectors, (ii) non-scalar shifted matrix, (iii) near-rank-one defect, (iv) nonzero reflection, (v) full-$S$ losslessness/reciprocity, and (vi) complex-$S$ convergence with increasing Fourier order. For simultaneous EPs **both** blocks must pass. For a stronger claim, demonstrate parameter-loop monodromy at converged order.

The saved matrices retain full complex phases, referenced to the same monitor planes. The common Jones phase $\tfrac12\arg\det R$ is undefined when $\det R=0$ and otherwise has a $\pi$ branch ambiguity; never use it as the sole EP observable.

In [ ]:
if best is not None:
    if hard_S:
        S_export=hard_S[max(hard_S)]
        d, near = report(S_export,'FINAL binary high-order candidate',target=TARGET)
        err=np.max(abs(hard_S[9]-hard_S[7]))
        # Both independently computed high-order matrices must satisfy the EP screen.
        pass7, _ = screen_candidate(diagnostics(hard_S[7]),target=TARGET)
        pass9, _ = screen_candidate(d,target=TARGET)
        converged=(err < .005) and pass7 and pass9
        near=bool(near and converged)
        print('Order 7→9 max complex matrix error:',err,'(limit: 0.005)')
        print('Screen at both high orders:',pass7,pass9)
        print('STATUS:', 'NUMERICAL NEAR-EP CANDIDATE' if near else
              'EP NOT VERIFIED (adjust geometry, convergence, or target)')
    else:
        S_export=best['S']
        near=False
        print('No hard-edge convergence; EP NOT VERIFIED.')
    R11,R22=reflection_blocks(S_export)
    for name,J in (('R11',R11),('R22',R22)):
        det=np.linalg.det(J)
        if abs(det)>1e-10:
            print(name,'half determinant phase modulo 180°:',np.angle(det,deg=True)/2)
        else:
            print(name,'determinant near zero: common phase ill-conditioned')
    p_export=best['params']  # Polished result is separate until validated at high order.
    path=OUT/f'{TARGET}_lossless_reflection_ep_candidate.npz'
    np.savez_compressed(path,S=S_export,R11_circular=R11,R22_circular=R22,
        params=json.dumps(p_export),kappa=np.array(0.),
        reference_planes=np.asarray(FIXED_PORTS),
        target=np.array(TARGET),verified_near_ep=np.array(near),
        binary_validated=np.array(bool(hard_S)))
    print('Saved complete complex scattering data:',path)
    print('Existing transmission-EP and full-S absorbing notebooks are untouched.')